%md
# 🚖 NYC Taxi Data Engineering Pipeline

**Author:** Srinath Yadav

## 📌 Project Overview

This project demonstrates an end-to-end Azure Data Engineering pipeline that ingests NYC Taxi trip data from a public API into Azure Data Lake Storage Gen2 (ADLS), processes it using Azure Databricks with PySpark, and implements the **Medallion Architecture (Bronze, Silver, and Gold)** for scalable and reliable data analytics.

---

## 🛠️ Technologies Used

- Azure Data Factory (ADF)
- Azure Data Lake Storage Gen2 (ADLS)
- Azure Databricks
- PySpark
- Delta Lake
- Medallion Architecture
- Git & GitHub

---

## 📂 Pipeline Flow

NYC Taxi API
⬇️

Azure Data Factory (Data Ingestion)
⬇️

ADLS Gen2 - Bronze Layer (Raw Data)
⬇️

Azure Databricks (Data Cleaning & Transformation)
⬇️

ADLS Gen2 - Silver Layer (Cleaned Data)
⬇️

Azure Databricks (Aggregation & Business Logic)
⬇️

ADLS Gen2 - Gold Layer (Analytics Ready Data)

---

## 🥉 Bronze Layer

- Ingest raw NYC Taxi data into ADLS.
- Preserve source data without transformations.

---

## 🥈 Silver Layer

- Remove duplicates.
- Handle null values.
- Standardize column names.
- Apply data type conversions.
- Add/drop/update columns based on requirements
- Perform data quality validations.

---

## 🥇 Gold Layer
- Converted data into delta format.
- Generate business-ready datasets.
- Create aggregated metrics.
- Optimize data for reporting and analytics.

---

## 🎯 Project Objective

Build a scalable and production-ready ETL pipeline using Azure services that demonstrates modern Data Engineering best practices, including orchestration, data lake architecture, Delta Lake, and Medallion Architecture.

In [0]:
storage_account = "nyctaxistoragesri"
tenant_id = "<tenant_id>"
client_id = "<client_id>"
client_secret = "<client_secret>"   # Retrieved from Databricks Secret Scope 

spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
spark.conf.get(
    "fs.azure.account.auth.type.nyctaxistoragesri.dfs.core.windows.net"
)

In [0]:
dbutils.fs.ls("abfss://bronze@nyctaxistoragesri.dfs.core.windows.net")

# 🥉 Bronze Layer

### Objective
Ingest raw NYC Taxi trip data from ADLS into Delta tables while preserving the source data for downstream processing.

### Trip Type



In [0]:
df_bronze = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load("abfss://bronze@nyctaxistoragesri.dfs.core.windows.net/trip_type")

In [0]:
df_bronze.display()

### Trip Zone

In [0]:
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://bronze@nyctaxistoragesri.dfs.core.windows.net/trip_zone")
df.display()

### TRIP DATA


In [0]:
df_trip_data  = spark.read.format("parquet")\
        .option("header",True)\
        .option("inferSchema",True)\
        .option("recursiveFileLookup",True)\
        .load("abfss://bronze@nyctaxistoragesri.dfs.core.windows.net/trip_data_2024")
        


# 🥈 Silver Layer

### Objective
Clean, validate, and transform the Bronze data by handling null values, removing duplicates, and applying business rules.


### TRIP TYPE

In [0]:
df_silver = df_bronze.withColumnRenamed("description","trip_description")

In [0]:
from pyspark.sql.functions import split, col, get

In [0]:
df_silver.write.mode("append")\
    .option("checkpointLocationn", "abfss://silver@nyctaxistoragesri.dfs.core.windows.net/checkpoints")\
    .format("parquet")\
    .option("path", "abfss://silver@nyctaxistoragesri.dfs.core.windows.net/trip_type")\
    .save()

### TRIP ZONE

In [0]:
df_trip_zone_silver = df.withColumn("zone1",split(col("Zone"),"/")[0])\
  .withColumn("zone2",get(split(col("Zone"),"/"),1))



In [0]:
df_trip_zone_silver.display()

In [0]:
df_trip_zone_silver.write.mode("append")\
    .option("checkpointLocationn", "abfss://silver@nyctaxistoragesri.dfs.core.windows.net/checkpoints")\
    .format("parquet")\
    .option("path", "abfss://silver@nyctaxistoragesri.dfs.core.windows.net/trip_zone")\
    .save()

##TRIP DATA - 2024


In [0]:
df_trip_data.display()

In [0]:
from pyspark.sql.functions import to_date, month, year

In [0]:
df_trip_data_silver = df_trip_data\
                        .withColumn("date", to_date(col("tpep_pickup_datetime")))\
                        .withColumn("month", month(col("tpep_pickup_datetime")))\
                        .withColumn("year", year(col("tpep_pickup_datetime")))


In [0]:
df_trip_data_silver.write.mode("append")\
    .option("checkpointLocationn", "abfss://silver@nyctaxistoragesri.dfs.core.windows.net/checkpoints")\
    .format("parquet")\
    .option("path", "abfss://silver@nyctaxistoragesri.dfs.core.windows.net/trip_data_2024")\
    .save()

## 🥇 Gold Layer

### Objective
Generate business-ready aggregated datasets and store them in **Delta format** within the Gold layer of Azure Data Lake Storage Gen2 for analytics and reporting.

### Actions Performed
- Aggregated the transformed Silver layer data.
- Generated business-ready metrics.
- Stored the final dataset in **Delta format** in the Gold layer.
- Prepared the data for downstream analytics and reporting.

### TRIP TYPE

In [0]:
df_trip_type  = spark.read.format("parquet")\
        .option("header",True)\
        .option("inferSchema",True)\
        .option("recursiveFileLookup",True)\
        .load("abfss://silver@nyctaxistoragesri.dfs.core.windows.net/trip_type")

In [0]:


df_trip_type.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@nyctaxistoragesri.dfs.core.windows.net/trip_type")

### TRIP ZONE

In [0]:
df_trip_zone  = spark.read.format("parquet")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("abfss://silver@nyctaxistoragesri.dfs.core.windows.net/trip_zone")

In [0]:
df_trip_type.write \
    .format("delta") \
    .mode("append") \
    .save("abfss://gold@nyctaxistoragesri.dfs.core.windows.net/trip_zone")

### TRIP DATA 2024

In [0]:
df_trip_data  = spark.read.format("parquet")\
        .option("header",True)\
        .option("recursiveFileLookup",True)\
        .option("inferSchema",True)\
        .load("abfss://silver@nyctaxistoragesri.dfs.core.windows.net/trip_data_2024")

In [0]:
df_trip_data.write \
    .format("delta") \
    .mode("append") \
    .save("abfss://gold@nyctaxistoragesri.dfs.core.windows.net/trip_data_2024")

## 